# Notebook 04: Binary Integer Programming Optimization

Selects the optimal hub locations from the predicted grid using BIP, verifies the separation constraint, and visualises the result.

## Load grid with predictions

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import joblib
import geopandas as gpd

gdf = gpd.read_file('../data/processed/grid.geojson')
print(f"Loaded grid: {len(gdf)} cells")
print(f"Columns: {list(gdf.columns)}")

# If p_profit is not present, generate it via the ensemble pipeline
if 'p_profit' not in gdf.columns:
    print("p_profit column missing — running ensemble to attach predictions...")
    from src.ensemble import generate_predictions
    from src.thompson_sampling import ThompsonSampler

    model = joblib.load('../models/lgbm_model.pkl')
    ts = ThompsonSampler(gdf['grid_id'].tolist())

    # Load augmented features
    features_df = joblib.load('../data/processed/features.pkl') if False else None
    X = joblib.load('../data/processed/X_train.pkl')
    from src.feature_engineering import FEATURE_COLUMNS
    feature_names = list(FEATURE_COLUMNS)

    # Add GWR columns if available
    if 'gwr_intercept' in gdf.columns:
        gwr_cols = gdf[['gwr_intercept', 'gwr_local_r2']].values.astype(np.float64)
        X = np.hstack([X, gwr_cols])
        feature_names = feature_names + ['gwr_intercept', 'gwr_local_r2']

    gdf = generate_predictions(gdf, model, ts, X, feature_names)
    gdf.to_file('../data/processed/grid.geojson', driver='GeoJSON')
    print(f"Predictions attached. p_profit mean = {gdf['p_profit'].mean():.4f}")
else:
    print(f"p_profit already present. mean = {gdf['p_profit'].mean():.4f}")

# Ensure centroid columns exist
if 'centroid_lat' not in gdf.columns:
    gdf['centroid_lat'] = gdf.geometry.centroid.y
if 'centroid_lon' not in gdf.columns:
    gdf['centroid_lon'] = gdf.geometry.centroid.x

gdf.head()

## Run BIP optimizer

In [ ]:
from src.bip_optimizer import run_bip

selected_ids, obj_value = run_bip(
    gdf,
    prob_col='p_profit',
    max_hubs=10,
    min_separation_km=2.0,
    min_prob_threshold=0.5,
)

print(f"Selected hubs: {selected_ids}")
print(f"Total objective score: {obj_value:.4f}")
print(f"Number of hubs selected: {len(selected_ids)}")

## Verify separation constraint

Compute pairwise Haversine distances between all selected hubs and confirm that every pair is at least 2 km apart.

In [ ]:
from src.bip_optimizer import _haversine_km

selected_rows = gdf[gdf['grid_id'].isin(selected_ids)]
lats = selected_rows['centroid_lat'].values.astype(float)
lons = selected_rows['centroid_lon'].values.astype(float)
ids = selected_rows['grid_id'].values.tolist()

min_dist = float('inf')
min_pair = (None, None)

print("Pairwise distances between selected hubs:")
for i in range(len(lats)):
    for j in range(i + 1, len(lats)):
        d = _haversine_km(lats[i], lons[i], lats[j], lons[j])
        print(f"  {ids[i]} ↔ {ids[j]}: {d:.2f} km")
        if d < min_dist:
            min_dist = d
            min_pair = (ids[i], ids[j])

print(f"\nMinimum pairwise distance: {min_dist:.2f} km ({min_pair[0]} ↔ {min_pair[1]})")

min_separation_km = 2.0
assert min_dist >= min_separation_km, (
    f"FAIL: minimum distance {min_dist:.2f} km < {min_separation_km} km"
)
print(f"✓ Separation constraint met (>= {min_separation_km} km)")

## Visualise selected hubs

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 1, figsize=(12, 12))

# Full grid in light grey
gdf.plot(ax=ax, color='lightgrey', edgecolor='white', linewidth=0.2)

# Selected hubs in red
hub_gdf = gdf[gdf['grid_id'].isin(selected_ids)]
hub_gdf.plot(ax=ax, color='red', edgecolor='darkred', linewidth=0.8)

# Annotate hub labels
for _, row in hub_gdf.iterrows():
    ax.annotate(
        row['grid_id'],
        xy=(row['centroid_lon'], row['centroid_lat']),
        fontsize=6,
        ha='center',
        color='white',
        fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.2', facecolor='red', alpha=0.8),
    )

ax.set_title(
    f'BIP Selected Hubs — Delhi ({len(selected_ids)} hubs, '
    f'obj = {obj_value:.3f})',
    fontsize=14,
)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
plt.tight_layout()
plt.show()